# Piano transcription benchmark on 10 MAESTRO v3 test recordings

Compare hFT-Transformer, Transkun V2, SFT-CRNN, HRT (Kong et al.), and Magenta Onsets and Frames.

Every model receives the same ten full recordings, in the same order. Reference notes use MIDI note-off times and are not extended by the sustain pedal. Quality is onset and onset+offset precision, recall, and F1 from `mir_eval`. Speed is wall-clock time from reading the audio file through the note list, after one warmup. Models run one at a time, each in its own process.

The local MAESTRO root, seed, and paths are set in the configuration cell. Author thresholds are fixed in `amt_benchmark.pins` before the timed run.


In [1]:
from pathlib import Path
import sys

REPO = Path.cwd().resolve()
if not (REPO / "amt_benchmark").exists():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))

MAESTRO_ROOT = Path("/Users/ilya/Documents/Samsung_project/data/maestro-v3.0.0")
THIRD_PARTY = Path("/Users/ilya/Documents/Samsung_project/data/amt-benchmark/third_party")
WEIGHTS = Path("/Users/ilya/Documents/Samsung_project/data/amt-benchmark/weights")
PYTHON = "/Users/ilya/Documents/Samsung_project/data/amt-benchmark/venv/bin/python"
RESULT_JSON = REPO / "results" / "maestro_v3_test10.json"
SEED = 20261123
DEVICE = "auto"


## Metric self-check

A synthetic note pair with a known onset match and a known onset+offset match is scored before any model runs.


In [2]:
from amt_benchmark.metrics import self_check

check = self_check()
check


{'passed': True,
 'case': 'synthetic_three_reference_notes',
 'onset': {'precision': 0.5,
  'recall': 0.6666666666666666,
  'f1': 0.5714285714285715},
 'onset_offset': {'precision': 0.25,
  'recall': 0.3333333333333333,
  'f1': 0.28571428571428575},
 'evaluation': {'event': '(midi_pitch, onset_s, offset_s)',
  'reference': 'pretty_midi note-on/note-off; CC64 sustain is not applied',
  'pedal_extension': False,
  'pitch_hz': '440 * 2 ** ((midi_pitch - 69) / 12)',
  'function': 'mir_eval.transcription.precision_recall_f1_overlap',
  'pitch_tolerance_cents': 50.0,
  'onset': {'onset_tolerance_s': 0.05,
   'offset_ratio': None,
   'pitch_tolerance_cents': 50.0},
  'onset_offset': {'onset_tolerance_s': 0.05,
   'offset_ratio': 0.2,
   'offset_min_tolerance_s': 0.05,
   'offset_tolerance': 'max(50 ms, 20% of the reference note duration)',
   'pitch_tolerance_cents': 50.0},
  'drop_nonpositive_duration': True}}

## Checkpoints and the ten recordings

Repository revisions, weight hashes, and inference settings are recorded before the timed run. The ten recordings are different test-split compositions, sampled with the fixed seed after the candidates are sorted.


In [3]:
import pandas as pd
from amt_benchmark.orchestrate import build_manifest, describe_checkpoints

manifest = build_manifest(MAESTRO_ROOT, seed=SEED)
checkpoints = describe_checkpoints(THIRD_PARTY, WEIGHTS)
pd.DataFrame(manifest["recordings"])[
    ["order", "canonical_composer", "canonical_title", "audio_filename", "audio_duration_s"]
]


,order,canonical_composer,canonical_title,audio_filename,audio_duration_s
0,0,Claude Debussy,"Etude ""Pour les accords""",2009/MIDI-Unprocessed_11_R1_2009_01-05_ORIG_MI...,271.703243
1,1,Claude Debussy,Jardins sous la pluie from Estampes,2011/MIDI-Unprocessed_04_R3_2011_MID--AUDIO_R3...,209.757528
2,2,Claude Debussy,Pagodas from Estampes No. 1,2008/MIDI-Unprocessed_05_R1_2008_01-04_ORIG_MI...,352.316168
3,3,Franz Liszt,"Réminiscences de Don Juan, S.418",2014/MIDI-UNPROCESSED_11-13_R1_2014_MID--AUDIO...,978.206032
4,4,Franz Schubert / Franz Liszt,"Song Transcriptions: Aufenthalt, Gretchen am S...",2004/MIDI-Unprocessed_XP_18_R1_2004_01-02_ORIG...,189.921474
5,5,Frédéric Chopin,Polonaise-Fantasie Op. 61,2008/MIDI-Unprocessed_12_R1_2008_01-04_ORIG_MI...,672.813968
6,6,Frédéric Chopin,"Valse brillante No. 1 in A-flat Major, Op. 34",2015/MIDI-Unprocessed_R1_D2-13-20_mid--AUDIO-f...,305.922925
7,7,Ludwig van Beethoven,"Sonata No. 16 in G Major, Op. 31 No. 1, First ...",2008/MIDI-Unprocessed_17_R1_2008_01-04_ORIG_MI...,278.578254
8,8,Robert Schumann,Toccata Op. 7,2006/MIDI-Unprocessed_06_R1_2006_01-04_ORIG_MI...,312.750227
9,9,Robert Schumann,"Toccata in C Major, Op. 7",2014/MIDI-UNPROCESSED_11-13_R1_2014_MID--AUDIO...,405.366327


In [4]:
rows = []
for model_id, item in checkpoints.items():
    weights = item.get("weights") or {}
    rows.append({
        "model": item["name"],
        "revision": item.get("revision"),
        "weight_file": weights.get("filename"),
        "sha256": weights.get("sha256"),
        "bytes": weights.get("bytes"),
        "available": item["available"],
        "reason": item.get("unavailable_reason"),
    })
pd.DataFrame(rows)


,model,revision,weight_file,sha256,bytes,available,reason
0,hFT-Transformer,226992b071b379b29232cd3df99c9b263cc8c312,model_016_003.pkl,f3a90cf0b82895f9923f3e766bff7e0a4759bc477370f0...,22136000.0,True,NaN
1,Transkun V2,c5cb8370e17b4a1650b5971ba87ee4b0d208e5b6,transkun_v2.pt,a7341e89ca3ceafce39c15ae1c4ea520b9d6d78498736d...,47830063.0,True,NaN
2,SFT-CRNN,21e69640b7c6c8a2860dab9a8b612e3bab18b87e,SFT-CRNN_Piano-MAESTRO_OTLoss_steps.340609.ckpt,0c065bc952591cda09197a2b811e0c117669ddba73982a...,61983435.0,True,NaN
3,HRT,566057930a8423aa31b91296e7002c513269446d,CRNN_note_F1=0.9677_pedal_F1=0.9186.pth,c3fa9730725bf4a762f1c14bc80cd5986eacda01b026f5...,171966578.0,True,NaN
4,OnF,NaN,NaN,NaN,NaN,False,weight file is missing: maestro_checkpoint.zip


## Smoke test

One available model transcribes the first selected recording. Model loading stays outside the timer. This checks the audio path before the ten-recording run.


In [5]:
from amt_benchmark.orchestrate import run_benchmark

smoke_model = next(model_id for model_id, item in checkpoints.items() if item["available"])
smoke = run_benchmark(
    MAESTRO_ROOT,
    THIRD_PARTY,
    WEIGHTS,
    PYTHON,
    WEIGHTS.parent / "worker-output" / "smoke.json",
    model_ids=[smoke_model],
    device=DEVICE,
    recording_limit=1,
    seed=SEED,
    cache_dir=WEIGHTS.parent / "worker-output",
)
smoke["models"][smoke_model]


running hft_transformer


finished hft_transformer status=complete n_success=1


{'model_id': 'hft_transformer',
 'name': 'hFT-Transformer',
 'status': 'complete',
 'n_success': 1,
 'n_expected': 1,
 'partial': False,
 'device': 'mps',
 'device_fallback': None,
 'dtype': 'torch.float32',
 'n_parameters': 5516574,
 'error': None,
 'macro_onset': {'precision': 0.9923479661699557,
  'recall': 0.8543689320388349,
  'f1': 0.9182038382709148},
 'macro_onset_offset': {'precision': 0.23640757148610553,
  'recall': 0.2035367545076283,
  'f1': 0.21874417738028695},
 'macro_over_n': 1,
 'median_latency_s': 32.71207216699986,
 'overall_rtf': 0.12039632597061084,
 'latency_sum_s': 32.71207216699986,
 'audio_duration_sum_s': 271.7032426303855,
 'peak_ram_bytes': 1473052672,
 'peak_ram_source': 'max of psutil.Process.memory_info().rss sampled every 100 ms by the parent and resource.getrusage(RUSAGE_SELF).ru_maxrss of the worker (bytes on macOS)',
 'peak_ram_reason': None,
 'peak_vram_bytes': 625207552,
 'peak_vram_source': 'peak of torch.mps.current_allocated_memory, sampled ever

## Full run

Each available model transcribes all ten recordings in one fresh process. A model that cannot load is recorded and the remaining models continue. Results are written to `results/maestro_v3_test10.json`.


In [6]:
document = run_benchmark(
    MAESTRO_ROOT,
    THIRD_PARTY,
    WEIGHTS,
    PYTHON,
    RESULT_JSON,
    model_ids=list(checkpoints),
    device=DEVICE,
    recording_limit=None,
    seed=SEED,
    cache_dir=WEIGHTS.parent / "worker-output",
)
document["speed_results_by_device"]


running hft_transformer


finished hft_transformer status=complete n_success=10


running transkun_v2


finished transkun_v2 status=complete n_success=10


running sft_crnn


finished sft_crnn status=failed n_success=0


... (3 outputs left out)

running onsets_frames


Python(10896) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


finished onsets_frames status=failed n_success=0


{'mps': ['hft_transformer', 'transkun_v2', 'hrt'],
 'not_run': ['sft_crnn', 'onsets_frames']}

## Results

The table is read back from the saved JSON. Speed numbers from different devices are not pooled; the device column keeps them separate.


In [7]:
import json
from amt_benchmark.orchestrate import summary_table

saved = json.loads(RESULT_JSON.read_text())
pd.DataFrame(summary_table(saved))


model,status,n_success,device,dtype,onset_f1,onset_precision,onset_recall,onset_offset_f1,onset_offset_precision,onset_offset_recall,median_latency_s,overall_rtf,peak_ram_bytes,peak_vram_bytes,n_parameters,weight_bytes
hFT-Transformer,complete,10,mps,torch.float32,0.958951,0.994172,0.926822,0.281863,0.291355,0.273124,39.844172,0.130965,3578642432,624683264.0,5516574.0,22136000.0
Transkun V2,complete,10,mps,torch.float32,0.973100,0.994245,0.953036,0.291500,0.297585,0.285706,30.126117,0.097643,1689157632,1141573376.0,12859535.0,47830063.0
SFT-CRNN,partial,3,mps,torch.float32,0.977046,0.990613,0.963945,0.400435,0.404750,0.396239,134.972958,0.685358,1891860480,NaN,13771106.0,61983435.0
HRT,complete,10,mps,torch.float32,0.949607,0.970113,0.930296,0.240501,0.245121,0.236127,135.988825,0.436387,2863104000,310479104.0,42946305.0,171966578.0
OnF,failed,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,212140032,NaN,NaN,NaN
